# Inspect DataLoader Batches

Replicates the exact train/validation loader setup from `train.py`, then samples one batch and shows:
- **Metadata** for each item (source file path, dataset label) — useful to check whether a batch is pulling consecutive crops from the same file
- **Decoded audio** for the target (`x`) and the timbre conditioning (`x_cond`)
- **Piano roll** for the structure conditioning (`x_time_cond`) when `structure_type == 'midi'`

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'after').exists():
    for parent in [ROOT, *ROOT.parents]:
        if (parent / 'after').exists():
            ROOT = parent
            break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Repository root: {ROOT}')

## Configuration — edit this cell

In [9]:
# ── paths ──────────────────────────────────────────────────────────────────
DB_PATHS = ["/data/nils/datasets/orchestral/emb_robert"
            ]  # one or more LMDB paths
CONFIG_FILE = "../after/diffusion/configs/midi.gin"
EMB_MODEL_PATH = "../pretrained/orchestral.ts"

# ── loader ─────────────────────────────────────────────────────────────────
WHICH_LOADER = "valid"  # "train" or "valid"
BATCH_SIZE = 4  # number of examples to show
GPU = 0  # -1 for CPU

# ── optional overrides (leave None to read from gin / auto-detect) ─────────
FILTER_INCLUDE = []  # e.g. ["jazz"]
FILTER_EXCLUDE = []
N_SIGNAL = 64

## Imports

In [ ]:
import gin
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import Audio, display

from after.dataset.dataset import SimpleDataset
from after.diffusion.utils import collate_fn_after, get_datasets

## Gin setup + model loading  *(mirrors `train.py`)*

In [ ]:
gin.clear_config()
gin.parse_config_files_and_bindings([CONFIG_FILE], [])

DEVICE = f"cuda:{GPU}" if GPU >= 0 and torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

emb_model = torch.jit.load(EMB_MODEL_PATH).to(DEVICE).eval()
try:
    audio_channels = emb_model.model.audio_channels
except:
    audio_channels = 1

dummy = torch.randn(1, audio_channels, 8192).cuda()
with torch.no_grad():
    z = emb_model.encode(dummy)
ae_ratio = dummy.shape[-1] // z.shape[-1]
ae_emb_size = z.shape[1]
print(f"ae_ratio={ae_ratio}  emb_size={ae_emb_size}")

SR = gin.query_parameter("%SR")
structure_type = gin.query_parameter("%STRUCTURE_TYPE")
print(f"SR={SR}  structure_type={structure_type}")

with gin.unlock_config():
    gin.bind_parameter("diffusion.utils.collate_fn_after.ae_ratio", ae_ratio)
    gin.bind_parameter("%IN_SIZE", ae_emb_size)
    gin.bind_parameter("diffusion.utils.get_datasets.db_list", DB_PATHS)
    gin.bind_parameter("diffusion.utils.get_datasets.emb_model_path",
                       EMB_MODEL_PATH)
    gin.bind_parameter("%N_SIGNAL", N_SIGNAL)
    if structure_type == "midi":
        gin.bind_parameter("diffusion.utils.get_datasets.ae_ratio", ae_ratio)
        gin.bind_parameter(
            "diffusion.utils.get_datasets.compress_tc",
            gin.query_parameter(
                "diffusion.utils.collate_fn_after.compress_tc"))
        gin.bind_parameter("diffusion.utils.get_datasets.sr", SR)
        gin.bind_parameter("diffusion.utils.collate_fn_after.precomp_pr",
                           False)

## Detect keys + build datasets

In [ ]:
allkeys = SimpleDataset(path=DB_PATHS[0]).get_keys()
structure_keys = [k for k in allkeys if "structure" in k and "midi" not in k]
timbre_keys = [k for k in allkeys if "timbre" in k]
midi_keys = [k for k in allkeys
             if "midi" in k] if structure_type == "midi" else []
data_keys = ["z"] + structure_keys + timbre_keys + midi_keys

print(f"structure_keys : {structure_keys}")
print(f"timbre_keys    : {timbre_keys}")
print(f"midi_keys      : {midi_keys}")
print(f"data_keys      : {data_keys}")

with gin.unlock_config():
    gin.bind_parameter("diffusion.utils.collate_fn_after.structure_keys",
                       structure_keys)
    gin.bind_parameter("diffusion.utils.collate_fn_after.timbre_keys",
                       timbre_keys)

filter_cfg = {"include": FILTER_INCLUDE, "exclude": FILTER_EXCLUDE}

# Build datasets — no key_sampler so every item carries full metadata for inspection
dataset, valset, train_sampler, val_sampler = get_datasets(
    data_keys=data_keys,
    use_cache=False,
    use_validation=True,
    filter=filter_cfg,
    # Deliberately omit structure_type so key_sampler stays None:
    # every item will load all keys, giving us metadata + all data.
)

print(f"\ntrain size: {len(dataset)}   val size: {len(valset)}")

## Helper functions

In [13]:
def decode_latent(z_np):
    """Decode a (C, T) numpy latent to a (samples,) float32 waveform."""
    z_t = torch.as_tensor(z_np, dtype=torch.float32,
                          device=DEVICE).unsqueeze(0)
    with torch.no_grad():
        audio = emb_model.decode(z_t)
    audio = audio.squeeze().cpu().numpy()
    if audio.ndim > 1:  # stereo → take first channel
        audio = audio[0]
    return audio.astype(np.float32)


def show_piano_roll(pr, compress_tc, title="Piano roll"):
    """pr: (128, T) float in [0, 1]."""
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.imshow(pr,
              aspect="auto",
              origin="lower",
              cmap="inferno",
              vmin=0,
              vmax=1,
              interpolation="nearest")
    ax.set_xlabel(f"Frame  (compress_tc={compress_tc})")
    ax.set_ylabel("MIDI pitch")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def show_item(i, raw_item, x, x_cond, x_time_cond):
    """Display one example from a batch."""
    meta = raw_item.get("metadata", {})
    path = meta.get("path", "(no path in metadata)")
    label = raw_item.get("label", meta.get("label", "?"))

    print("─" * 70)
    print(f"  Item {i}")
    print(f"  dataset : {label}")
    print(f"  path    : {Path(path).name}")
    print(f"  full    : {path}")
    print(
        f"  x shape : {tuple(x.shape)}   x_cond shape : {tuple(x_cond.shape)}")

    # ── audio ──────────────────────────────────────────────────────────────
    target_audio = decode_latent(x.numpy())
    timbre_audio = decode_latent(x_cond.numpy())

    print(f"  Target audio ({len(target_audio)/SR:.2f}s):")
    display(Audio(target_audio, rate=SR))

    print(f"  Timbre conditioning audio ({len(timbre_audio)/SR:.2f}s):")
    display(Audio(timbre_audio, rate=SR))

    # ── piano roll ─────────────────────────────────────────────────────────
    if structure_type == "midi":
        try:
            compress_tc = gin.query_parameter(
                "diffusion.utils.collate_fn_after.compress_tc")
        except:
            compress_tc = "?"
        pr = x_time_cond.numpy()  # (128, T)
        show_piano_roll(pr,
                        compress_tc,
                        title=f"Item {i} — piano roll  |  {Path(path).name}")
    print()

## Sample one batch and inspect

We pull raw items (keeping metadata), then run `collate_fn_after` manually on them so we get both the metadata and the tensors that the model actually sees.

In [ ]:
import itertools

ds = dataset if WHICH_LOADER == "train" else valset
smplr = train_sampler if WHICH_LOADER == "train" else val_sampler

print(
    f"Sampling {BATCH_SIZE} items from the '{WHICH_LOADER}' set ({len(ds)} examples)...\n"
)

# Draw indices the same way the DataLoader would
indices = list(itertools.islice(iter(smplr), BATCH_SIZE))
raw_batch = [ds[i] for i in indices]

# ── Consecutive-file check ─────────────────────────────────────────────────
paths = [item.get("metadata", {}).get("path", "?") for item in raw_batch]
labels = [item.get("label", "?") for item in raw_batch]

print("=== Source files for this batch ===")
for i, (idx, p, lbl) in enumerate(zip(indices, paths, labels)):
    print(f"  [{i}] sampler_idx={idx:6d}  dataset={lbl}  file={Path(p).name}")

n_unique = len(set(paths))
if n_unique < BATCH_SIZE:
    print(f"\n  ⚠  Only {n_unique}/{BATCH_SIZE} unique source files — "
          "some items are crops of the same track.")
else:
    print(f"\n  ✓  All {BATCH_SIZE} items come from different files.")

# ── Collate ────────────────────────────────────────────────────────────────
collated = collate_fn_after(raw_batch)
x = collated["x"]  # (B, C, T)
x_cond = collated["x_cond"]  # (B, C, T)
x_time_cond = collated["x_time_cond"]  # (B, 128, T*tc) or (B, C, T)

print(f"\nCollated shapes:")
print(f"  x            : {tuple(x.shape)}")
print(f"  x_cond       : {tuple(x_cond.shape)}")
print(f"  x_time_cond  : {tuple(x_time_cond.shape)}")

In [ ]:
for i, raw_item in enumerate(raw_batch):
    show_item(i, raw_item, x[i], x_cond[i], x_time_cond[i])

## Batch diversity — sampler index distribution

Draw a larger set of sampler indices and visualise how they're distributed across the dataset.
If the validation sampler concentrates on a narrow range of indices, you'll see crops from the same files.

In [ ]:
N_CHECK = min(256, len(ds))

idx_sample = list(itertools.islice(iter(smplr), N_CHECK))

fig, ax = plt.subplots(figsize=(14, 3))
ax.hist(idx_sample, bins=80, color="steelblue", edgecolor="none")
ax.set_xlabel("Dataset index")
ax.set_ylabel("Count")
ax.set_title(
    f"{WHICH_LOADER} sampler — index distribution over first {N_CHECK} draws")
plt.tight_layout()
plt.show()

# Source-file diversity over those N_CHECK draws
sampled_paths = [
    ds[i].get("metadata", {}).get("path", "?") for i in idx_sample
]
unique_files = len(set(sampled_paths))
print(f"Unique source files in {N_CHECK} draws: {unique_files}  "
      f"({100*unique_files/N_CHECK:.1f}% diversity)")

# How many consecutive pairs share the same source file?
consec = sum(1 for a, b in zip(sampled_paths, sampled_paths[1:]) if a == b)
print(f"Consecutive same-file pairs: {consec}/{N_CHECK-1} "
      f"({100*consec/(N_CHECK-1):.1f}%)")